<a href="https://colab.research.google.com/github/guadalupe-2406/-analysis-everpeak/blob/main/S12_Estudiante_Proyecto_Final_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sección nueva

# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python)

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [ ]:
# importar librerías
import pandas as pd

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
print("Descargando y cargando datasets desde los servidores de AWS...")

Descargando y cargando datasets desde los servidores de AWS...


In [ ]:


orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')



In [ ]:
print(" ¡Los tres DataFrames se cargaron perfectamente en memoria!\n")

 ¡Los tres DataFrames se cargaron perfectamente en memoria!



In [ ]:
print(f"Dataset 'orders':    {orders.shape[0]} registros y {orders.shape[1]} columnas.")
print(f"Dataset 'catalog':   {catalog.shape[0]} registros y {catalog.shape[1]} columnas.")
print(f"Dataset 'marketing': {marketing.shape[0]} registros y {marketing.shape[1]} columnas.")

Dataset 'orders':    25100 registros y 12 columnas.
Dataset 'catalog':   7 registros y 4 columnas.
Dataset 'marketing': 1620 registros y 5 columnas.


In [ ]:
orders.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas

---

In [ ]:
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'])
marketing['fecha'] = pd.to_datetime(marketing['fecha'])

In [ ]:
print("Fechas convertidas con éxito. Tipos de datos actuales:")
print(f"-> orders['fecha_hora_pedido']: {orders['fecha_hora_pedido'].dtype}")
print(f"-> marketing['fecha']: {marketing['fecha'].dtype}\n")

Fechas convertidas con éxito. Tipos de datos actuales:
-> orders['fecha_hora_pedido']: datetime64[ns]
-> marketing['fecha']: datetime64[ns]



In [ ]:
errores_cantidad = (orders['cantidad'] <= 0).sum()
errores_precio = (orders['precio_unitario'] <= 0).sum()

In [ ]:
# Verificar consistencia matemática del monto total
monto_calculado = (orders['cantidad'] * orders['precio_unitario']) - orders['monto_descuento']
# Buscamos filas donde la diferencia sea mayor a un centavo debido a decimales
descuadres = (abs(orders['monto_total'] - monto_calculado) > 0.01).sum()

In [ ]:
print(" REVISIÓN NUMÉRICA EN ORDERS:")
print(f"-> Cantidades <= 0: {errores_cantidad}")
print(f"-> Precios Unitarios <= 0: {errores_precio}")
print(f"-> Registros con montos totales descuadrados: {descuadres}\n")

 REVISIÓN NUMÉRICA EN ORDERS:
-> Cantidades <= 0: 4
-> Precios Unitarios <= 0: 0
-> Registros con montos totales descuadrados: 1130



In [ ]:
filas_orders_antes = orders.shape[0]
filas_mkt_antes = marketing.shape[0]

In [ ]:
orders = orders.drop_duplicates()
marketing = marketing.drop_duplicates()

print(" ELIMINACIÓN DE DUPLICADOS:")
print(f"-> Filas eliminadas en Orders: {filas_orders_antes - orders.shape[0]}")
print(f"-> Filas eliminadas en Marketing: {filas_mkt_antes - marketing.shape[0]}\n")

 ELIMINACIÓN DE DUPLICADOS:
-> Filas eliminadas en Orders: 0
-> Filas eliminadas en Marketing: 0



In [ ]:
orders['pais'] = orders['pais'].str.strip().str.capitalize()
marketing['pais'] = marketing['pais'].str.strip().str.capitalize()
print(f"-> Países únicos en Orders: {orders['pais'].unique()}")
print(f"-> Países únicos en Marketing: {marketing['pais'].unique()}")

-> Países únicos en Orders: ['Argentina' 'Mexico' 'Colombia']
-> Países únicos en Marketing: ['Mexico' 'Colombia' 'Argentina']


In [ ]:
orders = orders.dropna(subset=['pais'])
print(f"-> Países únicos en Orders ahora: {orders['pais'].unique()}")
print(f"-> Total de filas remanentes en Orders: {orders.shape[0]}")

-> Países únicos en Orders ahora: ['Argentina' 'Mexico' 'Colombia']
-> Total de filas remanentes en Orders: 24700


In [ ]:
orders_with_cost = pd.merge(
    orders,
    catalog[['nombre_producto', 'costo_unitario', 'proveedor']],
    on='nombre_producto',
    how='left'
)

# Calculamos el costo total y la ganancia neta por cada orden
orders_with_cost['costo_total'] = orders_with_cost['cantidad'] * orders_with_cost['costo_unitario']
orders_with_cost['ganancia_neta'] = orders_with_cost['monto_total'] - orders_with_cost['costo_total']

print("¡Tablas unificadas con éxito!")
print(f"Dimensiones del nuevo DataFrame 'orders_with_cost': {orders_with_cost.shape[0]} filas.\n")

# Ver una muestra de los resultados financieros calculados
orders_with_cost[['id_pedido', 'nombre_producto', 'cantidad', 'monto_total', 'costo_unitario', 'costo_total', 'ganancia_neta']].head()

¡Tablas unificadas con éxito!
Dimensiones del nuevo DataFrame 'orders_with_cost': 24700 filas.



,id_pedido,nombre_producto,cantidad,monto_total,costo_unitario,costo_total,ganancia_neta
0,order_0,Jacket-Winter-M,2.0,665.37,189.31,378.62,286.75
1,order_1,Tablet-Standard-64GB,1.0,171.86,25.21,25.21,146.65
2,order_2,Blender-XL-Red,2.0,195.99,176.64,353.28,-157.29
3,order_3,Tablet-Standard-64GB,1.0,242.87,25.21,25.21,217.66
4,order_4,Blender-XL-Red,1.0,336.28,176.64,176.64,159.64


In [ ]:
revenue_total = orders_with_cost['monto_total'].sum()
costo_total_productos = orders_with_cost['costo_total'].sum()
ganancia_bruta = orders_with_cost['ganancia_neta'].sum()
gasto_marketing_total = marketing['gasto'].sum()
margen_bruto_porcentaje = (ganancia_bruta / revenue_total) * 100
roi_global = ((ganancia_bruta - gasto_marketing_total) / gasto_marketing_total) * 100
print(f"Revenue Total (Ventas):      ${revenue_total:,.2f}")
print(f"Costo de Productos Vendidos: ${costo_total_productos:,.2f}")
print(f"Ganancia Bruta Operativa:   ${ganancia_bruta:,.2f}")
print(f"Margen Bruto (%):            {margen_bruto_porcentaje:.2f}%")
print(f"Inversión en Marketing:      ${gasto_marketing_total:,.2f}")
print(f" ROI Global del Negocio:      {roi_global:.2f}%")


Revenue Total (Ventas):      $51,876,621.51
Costo de Productos Vendidos: $43,080,359.27
Ganancia Bruta Operativa:   $8,762,371.62
Margen Bruto (%):            16.89%
Inversión en Marketing:      $2,871,843.53
 ROI Global del Negocio:      205.11%


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [ ]:
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)?
- ¿Cuál es el costo total?
- ¿Cuánto se ha invertido en marketing?
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden?
- ¿Cuál es la cantidad promedio de productos por orden?
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal?

In [ ]:
# 1. Ticket promedio por orden
ticket_promedio = orders_with_cost.groupby('id_pedido')['monto_total'].sum().mean()

# 2. Cantidad promedio de productos por orden
cant_promedio_por_orden = orders_with_cost.groupby('id_pedido')['cantidad'].sum().mean()

# 3. Producto más vendido
producto_mas_vendido = orders_with_cost.groupby('nombre_producto')['cantidad'].sum().idxmax()
unidades_producto_mas_vendido = orders_with_cost.groupby('nombre_producto')['cantidad'].sum().max()

# 4. Inversión en marketing por canal
mkt_por_canal = marketing.groupby('canal')['gasto'].sum().reset_index()

print("¡Parte 2 calculada con éxito!")


¡Parte 2 calculada con éxito!


In [ ]:
# CÁLCULO
revenue = orders_with_cost['monto_total'].sum()
costo_total_prod = orders_with_cost['costo_total'].sum()
inversion_marketing = marketing['gasto'].sum()
profit = revenue - costo_total_prod - inversion_marketing

# IMPRESIÓN
print(f"Ingreso Total (Revenue):   ${revenue:,.2f}")
print(f"Costo Total de Productos: ${costo_total_prod:,.2f}")
print(f"Inversión en Marketing:   ${inversion_marketing:,.2f}")
print(f"Ganancia Neta (Profit):   ${profit:,.2f}")
print(f"¿Es rentable?:            {'SÍ, ¡es rentable!' if profit > 0 else 'No'}")

Ingreso Total (Revenue):   $51,876,621.51
Costo Total de Productos: $43,080,359.27
Inversión en Marketing:   $2,871,843.53
Ganancia Neta (Profit):   $5,924,418.71
¿Es rentable?:            SÍ, ¡es rentable!


In [ ]:
print(f"Ticket Promedio por Orden:        ${ticket_promedio:.2f}")
print(f"Productos Promedio por Orden:    {cant_promedio_por_orden:.2f} unidades")
print(f"Productos Más Vendido:             {producto_mas_vendido} ({unidades_producto_mas_vendido:,.0f} un.)")

Ticket Promedio por Orden:        $2100.27
Productos Promedio por Orden:    7.17 unidades
Productos Más Vendido:             Laptop-Gaming-16GB (144,161 un.)


In [ ]:
for index, row in mkt_por_canal.iterrows():
    print(f"-> Canal '{row['canal']}': ${row['gasto']:,.2f}")

-> Canal 'organic': $913,533.01
-> Canal 'paid_search': $863,088.21
-> Canal 'social': $918,043.21


## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [ ]:

# 1. Consulta SQL corregida quitando nulos de raíz
query_funnel_clean = """
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
WHERE nombre_evento IS NOT NULL AND nombre_evento != 'NaN'
GROUP BY nombre_evento
"""
df_events_clean = pd.read_sql(query_funnel_clean, engine)
eventos_reales = df_events_clean['nombre_evento'].unique().tolist()
df_funnel_clean = df_events_clean.sort_values(by='usuarios_unicos', ascending=False).reset_index(drop=True)
df_funnel_clean['usuarios_paso_anterior'] = df_funnel_clean['usuarios_unicos'].shift(1)
df_funnel_clean['conversion_paso_a_paso_%'] = (df_funnel_clean['usuarios_unicos'] / df_funnel_clean['usuarios_paso_anterior']) * 100
df_funnel_clean['conversion_paso_a_paso_%'] = df_funnel_clean['conversion_paso_a_paso_%'].fillna(100.0)
usuarios_max = df_funnel_clean.loc[0, 'usuarios_unicos']
df_funnel_clean['conversion_global_%'] = (df_funnel_clean['usuarios_unicos'] / usuarios_max) * 100
df_funnel_clean['usuarios_perdidos'] = df_funnel_clean['usuarios_paso_anterior'] - df_funnel_clean['usuarios_unicos']
df_funnel_clean['usuarios_perdidos'] = df_funnel_clean['usuarios_perdidos'].fillna(0).astype(int)
df_perdidas = df_funnel_clean.drop(0)

etapa_critica_real = df_perdidas.loc[df_perdidas['usuarios_perdidos'].idxmax()]
print(df_funnel_clean[['nombre_evento', 'usuarios_unicos', 'conversion_paso_a_paso_%', 'conversion_global_%']].to_string(index=False, formatters={
    'usuarios_unicos': '{:,.0f}'.format,
    'conversion_paso_a_paso_%': '{:.2f}%'.format,
    'conversion_global_%': '{:.2f}%'.format
}))
print(f"¿Dónde se pierden más usuarios?: En la transición hacia '{etapa_critica_real['nombre_evento']}'")
print(f" Se pierden exactamente {etapa_critica_real['usuarios_perdidos']:,} usuarios en este paso.")
print(f"Tasa de conversión final (de principio a fin): {df_funnel_clean.iloc[-1]['conversion_global_%']:.2f}%")


   nombre_evento usuarios_unicos conversion_paso_a_paso_% conversion_global_%
     first_visit           7,796                  100.00%             100.00%
     add_to_cart           7,634                   97.92%              97.92%
     select_item           7,582                   99.32%              97.26%
  begin_checkout           7,208                   95.07%              92.46%
add_payment_info           6,250                   86.71%              80.17%
        purchase           6,240                   99.84%              80.04%
¿Dónde se pierden más usuarios?: En la transición hacia 'add_payment_info'
 Se pierden exactamente 958 usuarios en este paso.
Tasa de conversión final (de principio a fin): 80.04%


In [ ]:

# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()


,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [ ]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
WHERE nombre_evento IS NOT NULL
GROUP BY nombre_evento;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios_unicos
0,add_payment_info,6250
1,add_to_cart,7634
2,begin_checkout,7208
3,first_visit,7796
4,purchase,6240
5,select_item,7582


In [ ]:

query_conversion = '''
WITH funnel_totals AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos,
        CASE
            WHEN nombre_evento = 'first_visit' THEN 1
            WHEN nombre_evento = 'add_to_cart' THEN 2
            WHEN nombre_evento = 'select_item' THEN 3
            WHEN nombre_evento = 'begin_checkout' THEN 4
            WHEN nombre_evento = 'add_payment_info' THEN 5
            WHEN nombre_evento = 'purchase' THEN 6
            ELSE 7
        END AS paso_orden
    FROM events
    WHERE nombre_evento IS NOT NULL
    GROUP BY nombre_evento
),
visit_total AS (
    SELECT usuarios_unicos AS total_inicial
    FROM funnel_totals
    WHERE nombre_evento = 'first_visit'
)
SELECT
    f.nombre_evento,
    f.usuarios_unicos,
    ROUND((f.usuarios_unicos::numeric / LAG(f.usuarios_unicos) OVER (ORDER BY f.paso_orden)) * 100, 2) AS conversion_paso_a_paso_porcentaje,
    ROUND((f.usuarios_unicos::numeric / v.total_inicial) * 100, 2) AS conversion_global_porcentaje
FROM funnel_totals f
CROSS JOIN visit_total v
ORDER BY f.paso_orden;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,nombre_evento,usuarios_unicos,conversion_paso_a_paso_porcentaje,conversion_global_porcentaje
0,first_visit,7796,NaN,100.00
1,add_to_cart,7634,97.92,97.92
2,select_item,7582,99.32,97.26
3,begin_checkout,7208,95.07,92.46
4,add_payment_info,6250,86.71,80.17
5,purchase,6240,99.84,80.04


## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users`
- `user_activity`

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:


# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)



,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [ ]:
# Retención por cohortes
query_cohort_retention_final = '''
WITH cohort_inicial AS (
    -- 1. Identificamos el mes de registro (cohorte) de cada usuario asegurando formato DATE
    SELECT
        id_usuario,
        CAST(fecha_registro AS DATE) AS fecha_reg,
        TO_CHAR(CAST(fecha_registro AS DATE), 'YYYY-MM') AS cohorte
    FROM users
),
actividad_semanal AS (
    -- 2. Calculamos a cuántas semanas de su registro ocurrió cada actividad
    SELECT
        ua.id_usuario,
        c.cohorte,
        CAST(ua.fecha_actividad AS DATE) AS fecha_act,
        -- Diferencia en días convertida a semanas completas (0, 1, 2, 3...)
        FLOOR((CAST(ua.fecha_actividad AS DATE) - c.fecha_reg) / 7) AS semana_relativa
    FROM user_activity ua
    JOIN cohort_inicial c ON ua.id_usuario = c.id_usuario
),
totales_cohorte AS (
    -- 3. Contamos cuántos usuarios iniciales tiene cada cohorte
    SELECT
        cohorte,
        COUNT(DISTINCT id_usuario) AS usuarios_iniciales
    FROM cohort_inicial
    GROUP BY cohorte
),
usuarios_por_semana AS (
    -- 4. Contamos usuarios únicos activos en las semanas 1, 2 y 3
    SELECT
        cohorte,
        COUNT(DISTINCT CASE WHEN semana_relativa = 1 THEN id_usuario END) AS retenido_w1,
        COUNT(DISTINCT CASE WHEN semana_relativa = 2 THEN id_usuario END) AS retenido_w2,
        COUNT(DISTINCT CASE WHEN semana_relativa = 3 THEN id_usuario END) AS retenido_w3
    FROM actividad_semanal
    GROUP BY cohorte
)
-- 5. Juntamos todo y calculamos los porcentajes finales solicitados
SELECT
    t.cohorte,
    t.usuarios_iniciales,
    u.retenido_w1,
    u.retenido_w2,
    u.retenido_w3,
    ROUND((u.retenido_w1::numeric / t.usuarios_iniciales) * 100, 2) AS semana_1,
    ROUND((u.retenido_w2::numeric / t.usuarios_iniciales) * 100, 2) AS semana_2,
    ROUND((u.retenido_w3::numeric / t.usuarios_iniciales) * 100, 2) AS semana_3
FROM totales_cohorte t
JOIN usuarios_por_semana u ON t.cohorte = u.cohorte
ORDER BY t.cohorte;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,usuarios_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01,1627,1627,1627,1627,100.0,100.0,100.0
1,2025-02,1444,1444,1444,1444,100.0,100.0,100.0
2,2025-03,1636,1636,1636,1636,100.0,100.0,100.0
3,2025-04,1606,1606,1606,1606,100.0,100.0,100.0
4,2025-05,1687,1687,1687,1687,100.0,100.0,100.0


<div class="alert alert-block alert-success">
<b>Comentario del revisor</b> <a class="tocSkip"></a><br>
<b>Éxito</b> - El análisis de cohortes está correctamente estructurado y permite evaluar la retención de usuarios a lo largo del tiempo, mostrando de forma clara el comportamiento de las distintas cohortes y la estabilidad de las tasas de permanencia entre semanas.
</div>

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:**

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**
3. $H_0$ (Hipótesis nula): No hay una diferencia estadísticamente significativa entre la tasa de conversión del grupo de control y la del grupo de tratamiento. La modificación de la UI en el checkout no tiene impacto en la conversión. ($Tasa_{Control} = Tasa_{Tratamiento}$)$H_1$
4. (Hipótesis alternativa): Existe una diferencia estadísticamente significativa entre las tasas de conversión de ambos grupos. La modificación de la UI en el checkout sí impacta en la conversión. ($Tasa_{Control} \neq Tasa_{Tratamiento}$)
Nivel de significancia ($\alpha$): $0.05$ ($5\%$)     
5. **Aplicar el test estadístico adecuado**
Justificación del test: Dado que la métrica objetivo es categórica binaria (conversión: 1 o 0) y estamos comparando dos grupos independientes (control vs. tratamiento), el test estadístico adecuado es la Prueba de Independencia Chi-cuadrado ($\chi^2$).
   
6. **Interpretar el resultado**
conclusión Criterio de decisión: Dado que el p-valor obtenido ($0.4319$) es considerablemente mayor que nuestro nivel de significancia establecido ($\alpha = 0.05$), no se rechaza la hipótesis nula ($H_0$).Conclusión del experimento: No existe evidencia estadística suficiente para afirmar que los cambios en la UI del checkout afecten la tasa de conversión de compra. Aunque el grupo de tratamiento mostró una conversión ligeramente superior ($16.29\%$) frente al de control ($15.69\%$), esta pequeña variación de $0.6\%$ es estadísticamente insignificante y puede atribuirse al azar.
7. **Recomendación de negocio** No se aconseja implementar la nueva interfaz de checkout de forma definitiva bajo la premisa de que aumentará las ventas, ya que el comportamiento de los usuarios se mantiene fundamentalmente igual.

Hipótesis estadística
Hipótesis nula): La modificación en la UI del checkout no tiene un impacto estadísticamente significativo en la tasa de conversión de compra. Las tasas de conversión de ambos grupos son iguales ($Tasa_{Control} = Tasa_{Tratamiento}$).$H_1$

(Hipótesis alternativa): La modificación en la UI del checkout sí tiene un impacto estadísticamente significativo en la tasa de conversión de compra. Las tasas de conversión de ambos grupos son diferentes ($Tasa_{Control} \neq Tasa_{Tratamiento}$).

Test estadístico: Prueba de Independencia Chi-cuadrado de Pearson ($\chi^2$). (Se elige este porque estamos comparando la relación entre dos variables categóricas: la variante asignada y si el usuario convirtió o no)

Nivel de significancia ($\alpha$): $0.05$ (5%).


In [ ]:
 ckeckout = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')

In [ ]:
 ckeckout.head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [ ]:

import pandas as pd
from scipy.stats import chi2_contingency
tabla_contingencia = pd.crosstab(ckeckout['variante'], ckeckout['convirtio'])
print("=== TABLA DE CONTINGENCIA ===")
print(tabla_contingencia)
print("\n" + "="*40 + "\n")
tasas_conversion = pd.crosstab(ckeckout['variante'], ckeckout['convirtio'], normalize='index') * 100
print("=== TASAS DE CONVERSIÓN (%) ===")
print(tasas_conversion)
print("\n" + "="*40 + "\n")

chi2, p_valor, dof, esperados = chi2_contingency(tabla_contingencia)
print(f"Estadístico Chi2: {chi2:.4f}")
print(f"p-valor: {p_valor:.4e}")



=== TABLA DE CONTINGENCIA ===
convirtio       0    1
variante              
control      4186  779
tratamiento  4215  820


=== TASAS DE CONVERSIÓN (%) ===
convirtio            0          1
variante                         
control      84.310171  15.689829
tratamiento  83.714002  16.285998


Estadístico Chi2: 0.6178
p-valor: 4.3187e-01


## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión.

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard